# Feature Engineering for Fraud Detection (Full Dataset)

In this notebook, we will engineer features for the synthetic financial fraud detection dataset. 
We process the **entire dataset** without filtering any transaction types to ensure the model learns the context of all transactions (such as PAYMENT, CASH_IN, DEBIT) and does not narrow down the problem domain. 
We will also add advanced sender and destination cumulative history, temporal velocity features, and balance discrepancy flags.

In [1]:
import pandas as pd
import numpy as np
import os
import time

In [2]:
# Load clean dataset
data_path = "Synthetic_Financial_datasets_log.csv"
print("Loading dataset...")
df = pd.read_csv(data_path)
print(f"Loaded {df.shape[0]:,} rows and {df.shape[1]} columns.")

Loading dataset...


Loaded 6,362,620 rows and 11 columns.


### 1. Data Sorting

We will sort the dataset chronologically by `step` to ensure that our cumulative features do not leak information from the future.

In [3]:
# Sort by step to ensure correct chronological order for time-series/cumulative features
df = df.sort_values('step').reset_index(drop=True)
print("Dataset sorted chronologically.")

Dataset sorted chronologically.


### 2. Categorical Variable Encoding & Merchant Indicator

We keep all transaction types. Since `type` is a categorical variable, we will perform **One-Hot Encoding** to convert it into 5 separate binary columns.
In addition, destinations starting with `'M'` represent Merchants. Since merchant accounts never have fraud in this dataset, and their destination balance is not tracked (always 0.0), flagging them with a binary indicator `is_merchant_dest` is a crucial pattern for the model.

In [4]:
# One-hot encode type column
df_encoded = pd.get_dummies(df, columns=['type'], prefix='type', dtype=int)
for col in ['type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']:
    df[col] = df_encoded[col]

# Flag Merchant destinations (nameDest starting with M)
df['is_merchant_dest'] = df['nameDest'].str.startswith('M').astype(int)

print("Transaction types and merchant flag encoded.")
print(df[['type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER', 'is_merchant_dest']].sum())

Transaction types and merchant flag encoded.
type_CASH_IN        1399284
type_CASH_OUT       2237500
type_DEBIT            41432
type_PAYMENT        2151495
type_TRANSFER        532909
is_merchant_dest    2151495
dtype: int64


### 3. Time-Based Features

The `step` column represents hours. We extract cyclical temporal components:
- `hour_of_day`: `step % 24` (Representing the hour of the day)
- `day_of_month`: `step // 24` (Representing the day index)
- `day_of_week`: `day_of_month % 7` (Representing the day of the week)

In [5]:
df['hour_of_day'] = df['step'] % 24
df['day_of_month'] = df['step'] // 24
df['day_of_week'] = df['day_of_month'] % 7
print(df[['step', 'hour_of_day', 'day_of_month', 'day_of_week']].head())

   step  hour_of_day  day_of_month  day_of_week
0     1            1             0            0
1     1            1             0            0
2     1            1             0            0
3     1            1             0            0
4     1            1             0            0


### 4. Balance Discrepancy & Basic Features

We calculate actual changes and discrepancies in balances between expected and actual outcomes:
- `isOrigBalanceEnough`: Whether sender has sufficient balance (`oldbalanceOrg >= amount`).
- `orig_balance_after_expected`: Expected sender balance (`oldbalanceOrg - amount`).
- `dest_balance_after_expected`: Expected recipient balance (`oldbalanceDest + amount`).
- `is_amount_equal_oldbalanceOrig`: Flags transactions that equal the entire sender balance (indicates account emptying).
- `amount_to_orig_ratio` & `amount_to_dest_ratio`: Ratios of amounts to starting balances.
- `is_full_balance_transfer`: Whether transaction transfers >= 95% of sender balance.
- `balance_drop_ratio`: Expected drop proportion of sender balance.
- `orig_balance_change` & `dest_balance_change`: Realized balance changes.
- `errorBalanceOrig` & `errorBalanceDest` (and absolute error versions): Tracks mathematical discrepancies in balance updates.
- `isOrigBalanceZero`, `isDestBalanceZero`, `isNewBalanceOrigZero`, `isNewBalanceDestZero`: Binary indicators for exactly zero balances.

In [6]:
# Sender balance checks
df['isOrigBalanceEnough'] = (df['oldbalanceOrg'] >= df['amount']).astype(int)
df['orig_balance_after_expected'] = df['oldbalanceOrg'] - df['amount']
df['dest_balance_after_expected'] = df['oldbalanceDest'] + df['amount']
df['is_amount_equal_oldbalanceOrig'] = (df['amount'] == df['oldbalanceOrg']).astype(int)

# Balance ratio features
df['amount_to_orig_ratio'] = df['amount'] / (df['oldbalanceOrg'] + 1e-5)
df['amount_to_dest_ratio'] = df['amount'] / (df['oldbalanceDest'] + 1e-5)
df['is_full_balance_transfer'] = (df['amount_to_orig_ratio'] >= 0.95).astype(int)
df['balance_drop_ratio'] = (df['oldbalanceOrg'] - df['newbalanceOrig']) / (df['oldbalanceOrg'] + 1e-5)

# Balance change features
df['orig_balance_change'] = df['oldbalanceOrg'] - df['newbalanceOrig']
df['dest_balance_change'] = df['newbalanceDest'] - df['oldbalanceDest']

# Balance errors
df['errorBalanceOrig'] = df['oldbalanceOrg'] - df['amount'] - df['newbalanceOrig']
df['errorBalanceDest'] = df['oldbalanceDest'] + df['amount'] - df['newbalanceDest']
df['orig_balance_change_abs_error'] = np.abs(df['errorBalanceOrig'])
df['dest_balance_change_abs_error'] = np.abs(df['errorBalanceDest'])

# Binary indicators for zero values
df['isOrigBalanceZero'] = (df['oldbalanceOrg'] == 0).astype(int)
df['isDestBalanceZero'] = (df['oldbalanceDest'] == 0).astype(int)
df['isNewBalanceOrigZero'] = (df['newbalanceOrig'] == 0).astype(int)
df['isNewBalanceDestZero'] = (df['newbalanceDest'] == 0).astype(int)

print("Balance and basic features computed successfully.")

Balance and basic features computed successfully.


### 5. Outliers and Transaction Group Indicators

- `is_large_amount`: Flags whether `amount` is greater than `Q3 + 1.5 * IQR` (outlier transaction value).
- `is_transfer_or_cashout`: Flags transactions of type `TRANSFER` or `CASH_OUT` where 100% of frauds reside.

In [7]:
# Q3 + 1.5*IQR for large amount
q1 = df['amount'].quantile(0.25)
q3 = df['amount'].quantile(0.75)
iqr = q3 - q1
threshold = q3 + 1.5 * iqr
df['is_large_amount'] = (df['amount'] > threshold).astype(int)

# Risky types check
df['is_transfer_or_cashout'] = df['type'].isin(['TRANSFER', 'CASH_OUT']).astype(int)

print(f"Amount outlier threshold: {threshold:,.2f}")
print(df[['is_large_amount', 'is_transfer_or_cashout']].sum())

Amount outlier threshold: 501,719.34
is_large_amount            338078
is_transfer_or_cashout    2770409
dtype: int64


### 6. Cumulative History and Temporal Velocity (Sender & Destination)

We compute cumulative history metrics for both sender and destination accounts up to the point of the current transaction, including temporal sliding window features:
- **Sender cumulative history:**
  - `orig_cum_count`: Previous transaction count for `nameOrig`.
  - `orig_cum_sum`: Previous transaction sum for `nameOrig`.
  - `orig_cum_avg`: Average transaction amount of previous transactions for `nameOrig`.
- **Destination cumulative history:**
  - `dest_cum_count`: Previous transaction count for `nameDest`.
  - `dest_cum_sum`: Previous transaction sum for `nameDest`.
  - `dest_cum_avg`: Average transaction amount of previous transactions for `nameDest`.
- **Sliding Window & Velocity Features:**
  - `dest_unique_orig_count`: Number of different senders who have sent money to the destination before the current transaction.
  - `dest_txn_last_24h`: Velocity check: Number of transactions the destination has received in the prior 24 hours.
  - `dest_amount_last_24h`: Velocity check: Total amount the destination has received in the prior 24 hours.
  - `is_first_dest_tx`: Flags whether it's the very first transaction received by this destination (`dest_cum_count == 0`).
- **Ratios:**
  - `amount_to_orig_avg_ratio`: `amount / (orig_cum_avg + epsilon)`
  - `amount_to_dest_avg_ratio` / `amount_to_dest_cum_avg_ratio`: `amount / (dest_cum_avg + epsilon)`

In [8]:
t0 = time.time()

# 1. Senders cumulative features
print("Computing senders cumulative history...")
df['orig_cum_count'] = df.groupby('nameOrig').cumcount()
df['orig_cum_sum'] = df.groupby('nameOrig')['amount'].cumsum() - df['amount']
df['orig_cum_avg'] = df['orig_cum_sum'] / (df['orig_cum_count'] + 1e-5)

# 2. Destinations cumulative features
print("Computing destinations cumulative history...")
df['dest_cum_count'] = df.groupby('nameDest').cumcount()
df['dest_cum_sum'] = df.groupby('nameDest')['amount'].cumsum() - df['amount']
df['dest_cum_avg'] = df['dest_cum_sum'] / (df['dest_cum_count'] + 1e-5)

# 3. Unique orig count per dest
print("Computing unique senders per destination...")
df['is_first_sender_dest'] = ~df.duplicated(subset=['nameDest', 'nameOrig'])
df['dest_unique_orig_count'] = df.groupby('nameDest')['is_first_sender_dest'].cumsum() - df['is_first_sender_dest']

# 4. Temporal sliding window (24h velocity) for destination
print("Computing 24-hour velocity features per destination...")
df['step_datetime'] = pd.to_datetime(df['step'], unit='h', origin='2026-01-01')
df_sorted = df.sort_values(['nameDest', 'step_datetime']).copy()
rolling = df_sorted.groupby('nameDest').rolling('24h', on='step_datetime', closed='left')
df_sorted['dest_txn_last_24h'] = rolling['amount'].count().fillna(0).values
df_sorted['dest_amount_last_24h'] = rolling['amount'].sum().fillna(0).values
df = df_sorted.sort_index()

# 5. Helper indicator and ratios
df['is_first_dest_tx'] = (df['dest_cum_count'] == 0).astype(int)
df['amount_to_orig_avg_ratio'] = df['amount'] / (df['orig_cum_avg'] + 1e-5)
df['amount_to_dest_avg_ratio'] = df['amount'] / (df['dest_cum_avg'] + 1e-5)
df['amount_to_dest_cum_avg_ratio'] = df['amount'] / (df['dest_cum_avg'] + 1e-5)

print(f"Computed cumulative and velocity features in {time.time() - t0:.2f} seconds.")
print(df[['nameDest', 'dest_unique_orig_count', 'dest_txn_last_24h', 'dest_amount_last_24h', 'is_first_dest_tx']].tail())

Computing senders cumulative history...


Computing destinations cumulative history...


Computing unique senders per destination...


Computing 24-hour velocity features per destination...


Computed cumulative and velocity features in 98.95 seconds.
            nameDest  dest_unique_orig_count  dest_txn_last_24h  \
6362615  C1850423904                       0                0.0   
6362616   C776919290                       1                0.0   
6362617  C1881841831                       0                0.0   
6362618  C1365125890                       2                0.0   
6362619   C873221189                      27                0.0   

         dest_amount_last_24h  is_first_dest_tx  
6362615                   0.0                 1  
6362616                   0.0                 0  
6362617                   0.0                 1  
6362618                   0.0                 0  
6362619                   0.0                 0  


### 7. Feature Analysis & Correlation

Let's check the correlation of our engineered features with the target variable `isFraud`.

In [9]:
features_to_check = [
    'amount', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER',
    'is_merchant_dest', 'hour_of_day', 'day_of_month', 'day_of_week',
    'isOrigBalanceEnough', 'orig_balance_after_expected', 'dest_balance_after_expected',
    'is_amount_equal_oldbalanceOrig', 'amount_to_orig_ratio', 'amount_to_dest_ratio',
    'is_full_balance_transfer', 'balance_drop_ratio', 'orig_balance_change', 'dest_balance_change',
    'errorBalanceOrig', 'errorBalanceDest', 'orig_balance_change_abs_error', 'dest_balance_change_abs_error',
    'isOrigBalanceZero', 'isDestBalanceZero', 'isNewBalanceOrigZero', 'isNewBalanceDestZero',
    'is_large_amount', 'is_transfer_or_cashout',
    'orig_cum_count', 'orig_cum_sum', 'orig_cum_avg', 
    'dest_cum_count', 'dest_cum_sum', 'dest_cum_avg',
    'amount_to_orig_avg_ratio', 'amount_to_dest_avg_ratio',
    'dest_unique_orig_count', 'dest_txn_last_24h', 'dest_amount_last_24h', 'is_first_dest_tx', 'amount_to_dest_cum_avg_ratio', 'isFraud'
]

corr = df[features_to_check].corr()['isFraud'].sort_values(ascending=False)
print("Correlation of engineered features with isFraud:")
print(corr)

Correlation of engineered features with isFraud:
isFraud                           1.000000
is_amount_equal_oldbalanceOrig    0.989029
orig_balance_change               0.362472
amount_to_dest_cum_avg_ratio      0.189472
amount_to_dest_avg_ratio          0.189472
amount_to_dest_ratio              0.180201
amount                            0.076688
amount_to_orig_avg_ratio          0.076621
is_large_amount                   0.066696
errorBalanceDest                  0.055120
type_TRANSFER                     0.053869
dest_balance_change_abs_error     0.053840
isOrigBalanceEnough               0.047785
is_transfer_or_cashout            0.040938
day_of_month                      0.032577
isNewBalanceOrigZero              0.029984
dest_balance_change               0.027028
is_full_balance_transfer          0.025132
isDestBalanceZero                 0.016471
is_first_dest_tx                  0.014279
errorBalanceOrig                  0.011283
type_CASH_OUT                     0.011256
isNew

### 8. Save Engineered Dataset

We will save the dataset containing only relevant identifiers, the target, and the engineered features as a compressed Parquet file.

In [10]:
columns_to_save = [
    'step', 'type', 'nameOrig', 'nameDest', 'isFraud',
    'amount', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER',
    'is_merchant_dest', 'hour_of_day', 'day_of_month', 'day_of_week',
    'isOrigBalanceEnough', 'orig_balance_after_expected', 'dest_balance_after_expected',
    'is_amount_equal_oldbalanceOrig', 'amount_to_orig_ratio', 'amount_to_dest_ratio',
    'is_full_balance_transfer', 'balance_drop_ratio', 'orig_balance_change', 'dest_balance_change',
    'errorBalanceOrig', 'errorBalanceDest', 'orig_balance_change_abs_error', 'dest_balance_change_abs_error',
    'isOrigBalanceZero', 'isDestBalanceZero', 'isNewBalanceOrigZero', 'isNewBalanceDestZero',
    'is_large_amount', 'is_transfer_or_cashout',
    'orig_cum_count', 'orig_cum_sum', 'orig_cum_avg',
    'dest_cum_count', 'dest_cum_sum', 'dest_cum_avg',
    'amount_to_orig_avg_ratio', 'amount_to_dest_avg_ratio',
    'dest_unique_orig_count', 'dest_txn_last_24h', 'dest_amount_last_24h', 'is_first_dest_tx', 'amount_to_dest_cum_avg_ratio'
]

output_file = "Synthetic_Financial_datasets_features.parquet"
print(f"Saving dataset to {output_file}...")
df[columns_to_save].to_parquet(output_file, index=False, engine='pyarrow')
print(f"Successfully saved {output_file} ({os.path.getsize(output_file)/1024/1024:.2f} MB).")

Saving dataset to Synthetic_Financial_datasets_features.parquet...


Successfully saved Synthetic_Financial_datasets_features.parquet (801.57 MB).
